In [2]:
from pyspark.sql import functions as F

TAXI_PATH = "Files/taxi/yellow_tripdata_{ym}.parquet"

REQUIRED = {
    "PULocationID": "int",
    "DOLocationID": "int",
    "payment_type": "long",
    "trip_distance": "double",
    "fare_amount": "double",
    "extra": "double",
    "mta_tax": "double",
    "tip_amount": "double",
    "tolls_amount": "double",
    "improvement_surcharge": "double",
    "total_amount": "double",
    "congestion_surcharge": "double",
    "Airport_fee": "double",
    "cbd_congestion_fee": "double",
    "request_source": "string",
}

def read_taxi_month(spark, ym):
    df = spark.read.parquet(TAXI_PATH.format(ym=ym))
    actual = { c.lower() : c for c in df.columns}
    exprs = []
    for name, dtype in REQUIRED.items():
        if name.lower() in actual:
            exprs.append(F.col(actual[name.lower()]).cast(dtype).alias(name))
        else:
            exprs.append(F.lit(None).cast(dtype).alias(name))
    adapted = df.select("tpep_pickup_datetime", "tpep_dropoff_datetime", *exprs)
    return adapted

StatementMeta(, 322c7ccd-7734-494c-b477-c8956656dd91, 4, Finished, Available, Finished, False)

In [3]:
from functools import reduce
from pyspark.sql import DataFrame
import re

QUANTILE = 0.999
QUANTILE_ERROR = 0.0001

taxi_file_names = notebookutils.fs.ls("Files/taxi")
pattern = r"yellow_tripdata_(\d{4}-\d{2})\.parquet"
matches = [re.search(pattern, f.name) for f in taxi_file_names]
TAXI_MONTHS = sorted(m.group(1) for m in matches if m)

silver_cols = [
    "tpep_pickup_datetime", "tpep_dropoff_datetime", "pickup_date", "pickup_hour",
    "PULocationID", "DOLocationID",
    "trip_distance", "duration_min", "is_zero_distance", "is_non_positive_amount",
    "is_invalid_duration","is_outlier_amount","is_outlier_duration",
    "fare_amount", "extra", "mta_tax", "tip_amount", "tolls_amount",
    "improvement_surcharge", "total_amount", "congestion_surcharge",
    "Airport_fee", "cbd_congestion_fee",
    "request_source", "payment_type"
]

def process_month(spark, ym):
    raw = read_taxi_month(spark, ym)
    start = F.to_date(F.lit(ym + "-01"))
    end = F.add_months(start, 1)
    in_range = (F.col("tpep_pickup_datetime") >= start) & (F.col("tpep_pickup_datetime") < end)

    df = raw.filter(in_range)
    rejected = raw.filter(~in_range).withColumn("reject_reason", F.lit("pickup_out_of_range"))

    df = df.withColumn("duration_min", F.timestamp_diff("SECOND", F.col("tpep_pickup_datetime"), F.col("tpep_dropoff_datetime")) / 60)
    df = df.withColumn("is_invalid_duration", F.col("duration_min") <= 0)
    df = df.withColumn("duration_min", F.when(~F.col("is_invalid_duration"), F.col("duration_min")))

    q = df.approxQuantile(["total_amount", "duration_min"], [QUANTILE], QUANTILE_ERROR)
    amount_cap = q[0][0]
    duration_cap = q[1][0]
    df = df.withColumn("is_outlier_amount", F.col("total_amount") > amount_cap)
    df = df.withColumn("is_outlier_duration", F.coalesce(F.col("duration_min") > duration_cap, F.lit(False)))


    df = df.withColumn("is_zero_distance", F.when(F.col("trip_distance") <= 0, True).otherwise(False))
    df = df.withColumn("is_non_positive_amount", F.when(F.col("total_amount") <= 0, True).otherwise(False))
    df = df.withColumns({"pickup_date": F.to_date(F.col("tpep_pickup_datetime")), "pickup_hour": F.hour(F.col("tpep_pickup_datetime"))})

    df = df.select(*silver_cols)

    return df, rejected

silver_parts = []
rejected_parts = []

for ym in TAXI_MONTHS:
    df_m, rej_m = process_month(spark, ym)
    silver_parts.append(df_m)
    rejected_parts.append(rej_m)

silver = reduce(DataFrame.unionByName, silver_parts)
rejected = reduce(DataFrame.unionByName, rejected_parts)
print(TAXI_MONTHS)

StatementMeta(, 322c7ccd-7734-494c-b477-c8956656dd91, 5, Finished, Available, Finished, False)

['2024-01', '2024-02', '2024-03', '2024-04', '2024-05', '2024-06', '2024-07', '2024-08', '2024-09', '2024-10', '2024-11', '2024-12', '2025-01', '2025-02', '2025-03', '2025-04', '2025-05', '2025-06', '2025-07', '2025-08', '2025-09', '2025-10', '2025-11', '2025-12', '2026-01', '2026-02', '2026-03', '2026-04', '2026-05', '2026-06', '2026-07', '2026-08', '2026-09']


In [12]:
rows_count = 0
null_date = 0
for ym in TAXI_MONTHS:
    df = spark.read.parquet(TAXI_PATH.format(ym=ym))
    rows_count += df.count()
    null_date += df.filter(F.col("tpep_pickup_datetime").isNull()).count()
print(rows_count, null_date)

StatementMeta(, 322c7ccd-7734-494c-b477-c8956656dd91, 14, Finished, Available, Finished, False)

123471721 0


In [7]:
silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("lh_silver.dbo.taxi")
rejected.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("lh_silver.dbo.taxi_rejected")
loaded_silver = spark.table("lh_silver.dbo.taxi")
loaded_rejected = spark.table("lh_silver.dbo.taxi_rejected")
print("bronze:", rows_count)
print("silver:", loaded_silver.count())
print("rejected:", loaded_rejected.count())

StatementMeta(, 322c7ccd-7734-494c-b477-c8956656dd91, 9, Finished, Available, Finished, False)

bronze: 127909452
silver: 123470919
rejected: 802
